# Standalone Integrated Memory V2.3 — Decision Model
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V23_Decision_Colab.ipynb)

This is the standalone successor to `Laya_Integrated_Memory_V22_Colab.ipynb`.

**Targets**
- no `laya` Python dependency in conversion, export, or inference;
- reconstruct the public typed-decision checkpoint directly from its config + safetensors;
- replace **every ModernBERT `full_attention` layer** with TinyCeNN Integrated Memory V2.3;
- keep sliding-attention layers unchanged;
- do a short layer-transfer warm-up followed by end-to-end decision-logit distillation;
- export the **entire model**, not an adapter;
- run a fast held-out teacher/student fidelity + latency evaluation and save the report;\n- save the complete standalone model to Hugging Face for later direct reload;\n- provide `runtime.decide(state, actions)` for later game-control integration.

V2.3 uses two non-quadratic global associative-memory routes (symmetric-softmax features and ELU+1 features), a CeNN-style local depthwise route, and a direct-value route. No global T×T attention score matrix is formed by the replacement.

In [1]:
#@title 1. Install
%pip -q install -U "transformers>=4.45" "datasets>=2.20" "huggingface_hub>=0.25" "safetensors>=0.4"
%pip -q install "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@codex/standalone-integrated-memory-v23"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
#@title 2. Configure and load source checkpoint WITHOUT Laya
from pathlib import Path
import copy, json, os, random, shutil
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from safetensors.torch import save_file
from transformers import AutoTokenizer

import tinycenn_lm.standalone_decision as sd
from tinycenn_lm.standalone_decision import (
    QTYPES, IntegratedMemoryV23Attention, build_checkpoint_model,
    build_sequence, collate_items, converted_full_attention_indices,
    full_attention_indices, load_standalone, normalize_question,
    replacement_trainable_parameters,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
FEATURE_DIM = 96 #@param {type:"integer"}
LOCAL_KERNEL = 5 #@param {type:"integer"}
LOCAL_STEPS = 120 #@param {type:"integer"}
GLOBAL_STEPS = 300 #@param {type:"integer"}
TRAIN_CASES = 700 #@param {type:"integer"}
VAL_CASES = 160 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Standalone_Integrated_Memory_V23_Decision")
HF_REPO_ID = "vtava/Laya-Integrated-Memory-V23-Decision" #@param {type:"string"}
UPLOAD_TO_HF = True #@param {type:"boolean"}
HF_PRIVATE = False #@param {type:"boolean"}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
use_amp = device.type == "cuda"

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student.to(device).eval().requires_grad_(False)

full_layers = full_attention_indices(student)
print("device:", device, "| source:", SOURCE_MODEL)
print("full-attention layers:", full_layers, "| count:", len(full_layers))
assert full_layers, "No full_attention layers detected."
assert "laya" not in globals(), "This notebook must not import Laya."


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

device: cuda | source: convaiinnovations/laya-typed-decisions
full-attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27] | count: 10


In [6]:
#@title 3. Build compact typed-decision train/validation items
def _jsonish(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except Exception: return x
    return x

def make_items(split, limit, seed):
    ds = list(load_dataset("LocalLLaMA/typed-decisions", "all", split=split))
    rng = random.Random(seed); rng.shuffle(ds); ds = ds[:min(limit, len(ds))]
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in ds:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(state_text.replace(tokenizer.mask_token, " "), add_special_tokens=False)["input_ids"]
        for qid, qdef in questions.items():
            try:
                q = normalize_question(qdef)
                ids, markers = build_sequence(
                    tokenizer, state, q,
                    max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                    head_max_len=head_max_len,
                    truncate_left=isinstance(state, list),
                    state_ids=state_ids,
                )
                if len(markers) >= 2:
                    out.append({"ids": ids, "markers": markers, "qtype": QTYPES[q["t"]], "qid": qid})
            except Exception:
                pass
    rng.shuffle(out)
    return out

train_items = make_items("train", TRAIN_CASES, SEED)
val_items = make_items("test", VAL_CASES, SEED + 1)
print("train items:", len(train_items), "| validation items:", len(val_items))
assert len(train_items) >= 64 and len(val_items) >= 32

def batch(items, offset, n=BATCH):
    selected = [items[(offset+i) % len(items)] for i in range(n)]
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(z.detach() for z in kwargs["position_embeddings"])
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            model(**b)
    finally:
        h1.remove(); h2.remove()
    return got

def local_loss(pred, target, valid):
    m = valid[:, :, None].float()
    mse = (((pred.float()-target.float())**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
    power = ((target.float()**2)*m).sum() / (m.sum()*target.shape[-1]).clamp_min(1)
    nmse = mse / power.clamp_min(1e-8)
    cos = F.cosine_similarity(pred.float()[valid], target.float()[valid], dim=-1).mean()
    return nmse + 1.10*(1-cos), nmse, cos


train items: 3500 | validation items: 800


In [8]:
#@title 4. Stage A — replace ALL full-attention layers and locally warm each replacement
local_report = {}
for layer_idx in full_layers:
    print(f"\n=== full-attention layer {layer_idx} ===")
    original = student.encoder.layers[layer_idx].attn
    repl = IntegratedMemoryV23Attention(original, feature_dim=FEATURE_DIM, local_kernel=LOCAL_KERNEL).to(device)
    params = repl.trainable_core_parameters()
    opt = torch.optim.AdamW(params, lr=1e-1, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, LOCAL_STEPS), eta_min=6e-4)
    best = None; best_score = float("inf")

    probe_b = batch(val_items, layer_idx*11, min(6, BATCH+2))
    probe = attn_io(student, layer_idx, probe_b)
    probe_valid = probe_b["attention_mask"].bool()

    for step in range(LOCAL_STEPS):
        b = batch(train_items, step*BATCH + layer_idx*17)
        target = attn_io(student, layer_idx, b)
        valid = b["attention_mask"].bool()
        repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            pred, _ = repl(target["x"], position_embeddings=target["pos"], attention_mask=target["mask"])
            loss, nmse, cos = local_loss(pred, target["y"], valid)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
        loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()

        if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
            repl.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                pp, _ = repl(probe["x"], position_embeddings=probe["pos"], attention_mask=probe["mask"])
                ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
            score = float(ploss)
            if score < best_score:
                best_score = score
                best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
            print(f"step {step+1:03d}/{LOCAL_STEPS} nmse={float(pnmse):.4f} cos={float(pcos):.4f}")

    if best is not None: repl.load_state_dict(best)
    repl.eval()
    student.encoder.layers[layer_idx].attn = repl
    local_report[str(layer_idx)] = {"best_local_score": best_score}
    if torch.cuda.is_available(): torch.cuda.empty_cache()

converted = converted_full_attention_indices(student)
print("\nconverted:", converted)
assert converted == full_layers, f"Every source full-attention layer must be replaced: {converted} != {full_layers}"



=== full-attention layer 0 ===
step 001/120 nmse=0.9487 cos=0.6550
step 030/120 nmse=0.0444 cos=0.9740
step 060/120 nmse=0.0166 cos=0.9899
step 090/120 nmse=0.0108 cos=0.9933
step 120/120 nmse=0.0100 cos=0.9938

=== full-attention layer 3 ===
step 001/120 nmse=1.1915 cos=0.5256
step 030/120 nmse=0.0571 cos=0.9643
step 060/120 nmse=0.0222 cos=0.9866
step 090/120 nmse=0.0144 cos=0.9914
step 120/120 nmse=0.0132 cos=0.9921

=== full-attention layer 6 ===
step 001/120 nmse=0.9231 cos=0.4452
step 030/120 nmse=0.3544 cos=0.8172
step 060/120 nmse=0.3050 cos=0.8448
step 090/120 nmse=0.2893 cos=0.8533
step 120/120 nmse=0.2837 cos=0.8561

=== full-attention layer 9 ===
step 001/120 nmse=1.0294 cos=0.3598
step 030/120 nmse=0.4608 cos=0.7518
step 060/120 nmse=0.4088 cos=0.7849
step 090/120 nmse=0.3867 cos=0.7989
step 120/120 nmse=0.3825 cos=0.8016

=== full-attention layer 12 ===
step 001/120 nmse=0.9085 cos=0.4117
step 030/120 nmse=0.4385 cos=0.7623
step 060/120 nmse=0.3532 cos=0.8091
step 090/12

In [9]:
#@title 4. Stage A — replace ALL full-attention layers and locally warm each replacement
local_report = {}
for layer_idx in full_layers:
    print(f"\n=== full-attention layer {layer_idx} ===")
    original = student.encoder.layers[layer_idx].attn
    repl = IntegratedMemoryV23Attention(original, feature_dim=FEATURE_DIM, local_kernel=LOCAL_KERNEL).to(device)
    params = repl.trainable_core_parameters()
    opt = torch.optim.AdamW(params, lr=1e-1, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, LOCAL_STEPS), eta_min=6e-4)
    best = None; best_score = float("inf")

    probe_b = batch(val_items, layer_idx*11, min(6, BATCH+2))
    probe = attn_io(student, layer_idx, probe_b)
    probe_valid = probe_b["attention_mask"].bool()

    for step in range(LOCAL_STEPS):
        b = batch(train_items, step*BATCH + layer_idx*17)
        target = attn_io(student, layer_idx, b)
        valid = b["attention_mask"].bool()
        repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            pred, _ = repl(target["x"], position_embeddings=target["pos"], attention_mask=target["mask"])
            loss, nmse, cos = local_loss(pred, target["y"], valid)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
        loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()

        if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
            repl.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                pp, _ = repl(probe["x"], position_embeddings=probe["pos"], attention_mask=probe["mask"])
                ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
            score = float(ploss)
            if score < best_score:
                best_score = score
                best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
            print(f"step {step+1:03d}/{LOCAL_STEPS} nmse={float(pnmse):.4f} cos={float(pcos):.4f}")

    if best is not None: repl.load_state_dict(best)
    repl.eval()
    student.encoder.layers[layer_idx].attn = repl
    local_report[str(layer_idx)] = {"best_local_score": best_score}
    if torch.cuda.is_available(): torch.cuda.empty_cache()

converted = converted_full_attention_indices(student)
print("\nconverted:", converted)
assert converted == full_layers, f"Every source full-attention layer must be replaced: {converted} != {full_layers}"



=== full-attention layer 0 ===
step 001/120 nmse=0.9632 cos=0.6595
step 030/120 nmse=0.0337 cos=0.9813
step 060/120 nmse=0.0095 cos=0.9943
step 090/120 nmse=0.0055 cos=0.9965
step 120/120 nmse=0.0050 cos=0.9968

=== full-attention layer 3 ===
step 001/120 nmse=1.1915 cos=0.5355
step 030/120 nmse=0.0409 cos=0.9751
step 060/120 nmse=0.0142 cos=0.9912
step 090/120 nmse=0.0080 cos=0.9950
step 120/120 nmse=0.0072 cos=0.9955

=== full-attention layer 6 ===
step 001/120 nmse=0.8778 cos=0.5475
step 030/120 nmse=0.0998 cos=0.9505
step 060/120 nmse=0.0469 cos=0.9772
step 090/120 nmse=0.0322 cos=0.9845
step 120/120 nmse=0.0294 cos=0.9860

=== full-attention layer 9 ===
step 001/120 nmse=1.0390 cos=0.4658
step 030/120 nmse=0.1278 cos=0.9347
step 060/120 nmse=0.0600 cos=0.9698
step 090/120 nmse=0.0430 cos=0.9783
step 120/120 nmse=0.0403 cos=0.9796

=== full-attention layer 12 ===
step 001/120 nmse=0.8816 cos=0.4930
step 030/120 nmse=0.1838 cos=0.9075
step 060/120 nmse=0.0770 cos=0.9590
step 090/12

In [10]:
#@title 4. Stage A — replace ALL full-attention layers and locally warm each replacement
local_report = {}
for layer_idx in full_layers:
    print(f"\n=== full-attention layer {layer_idx} ===")
    original = student.encoder.layers[layer_idx].attn
    repl = IntegratedMemoryV23Attention(original, feature_dim=FEATURE_DIM, local_kernel=LOCAL_KERNEL).to(device)
    params = repl.trainable_core_parameters()
    opt = torch.optim.AdamW(params, lr=1e-1, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, LOCAL_STEPS), eta_min=6e-4)
    best = None; best_score = float("inf")

    probe_b = batch(val_items, layer_idx*11, min(6, BATCH+2))
    probe = attn_io(student, layer_idx, probe_b)
    probe_valid = probe_b["attention_mask"].bool()

    for step in range(LOCAL_STEPS):
        b = batch(train_items, step*BATCH + layer_idx*17)
        target = attn_io(student, layer_idx, b)
        valid = b["attention_mask"].bool()
        repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            pred, _ = repl(target["x"], position_embeddings=target["pos"], attention_mask=target["mask"])
            loss, nmse, cos = local_loss(pred, target["y"], valid)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
        loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()

        if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
            repl.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                pp, _ = repl(probe["x"], position_embeddings=probe["pos"], attention_mask=probe["mask"])
                ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
            score = float(ploss)
            if score < best_score:
                best_score = score
                best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
            print(f"step {step+1:03d}/{LOCAL_STEPS} nmse={float(pnmse):.4f} cos={float(pcos):.4f}")

    if best is not None: repl.load_state_dict(best)
    repl.eval()
    student.encoder.layers[layer_idx].attn = repl
    local_report[str(layer_idx)] = {"best_local_score": best_score}
    if torch.cuda.is_available(): torch.cuda.empty_cache()

converted = converted_full_attention_indices(student)
print("\nconverted:", converted)
assert converted == full_layers, f"Every source full-attention layer must be replaced: {converted} != {full_layers}"



=== full-attention layer 0 ===
step 001/120 nmse=0.9746 cos=0.6622


KeyboardInterrupt: 

In [12]:
#@title 5. Stage B — short end-to-end decision recovery fine-tune
for p in student.parameters(): p.requires_grad = False
for p in replacement_trainable_parameters(student): p.requires_grad = True
for module in (student.head, student.type_emb, student.scorer, student.act_head):
    if module is not None:
        for p in module.parameters(): p.requires_grad = True

trainable = [(n,p) for n,p in student.named_parameters() if p.requires_grad]
print("trainable params:", f"{sum(p.numel() for _,p in trainable):,}")
opt = torch.optim.AdamW([p for _,p in trainable], lr=2.5e-2, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1,GLOBAL_STEPS), eta_min=2e-5)
T = 1.5

def distill_loss(slog, tlog, mask, sact, tact):
    s = slog.masked_fill(~mask, -1e4)/T
    t = tlog.masked_fill(~mask, -1e4)/T
    tp = torch.softmax(t.detach(), -1)
    kl = (tp*(torch.log_softmax(t.detach(),-1)-torch.log_softmax(s,-1))).sum(-1).mean()*(T*T)
    ce = F.cross_entropy(slog, tlog.argmax(-1))
    ap = torch.softmax(tact.detach()/T,-1)
    akl = (ap*(torch.log_softmax(tact.detach()/T,-1)-torch.log_softmax(sact/T,-1))).sum(-1).mean()*(T*T)
    return kl + .15*ce + .05*akl

@torch.no_grad()
def validate(limit=96):
    teacher.eval(); student.eval(); total=agree=0; kls=[]
    for off in range(0, min(limit,len(val_items)), BATCH):
        b = batch(val_items, off, min(BATCH, min(limit,len(val_items))-off))
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            tl,_ = teacher(**b); sl,_ = student(**b)
        m=b["marker_mask"].bool(); tm=tl.masked_fill(~m,-1e4); sm=sl.masked_fill(~m,-1e4)
        agree += int((tm.argmax(-1)==sm.argmax(-1)).sum()); total += tl.shape[0]
        tp=torch.softmax(tm/T,-1)
        kls.append(float((tp*(torch.log_softmax(tm/T,-1)-torch.log_softmax(sm/T,-1))).sum(-1).mean()))
    return {"agreement":agree/max(1,total),"mean_kl":sum(kls)/max(1,len(kls))}

student.eval()
best_state=None; best_score=float("inf"); best_metrics=None
eval_every=max(25, GLOBAL_STEPS//6)
for step in range(GLOBAL_STEPS):
    b=batch(train_items, step*BATCH+31)
    opt.zero_grad(set_to_none=True)
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        tl,ta=teacher(**b)
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        sl,sa=student(**b); loss=distill_loss(sl,tl,b["marker_mask"].bool(),sa,ta)
    loss.backward(); torch.nn.utils.clip_grad_norm_([p for _,p in trainable],1.0); opt.step(); sched.step()

    if step==0 or (step+1)%eval_every==0 or step+1==GLOBAL_STEPS:
        m=validate(); score=m["mean_kl"]+.35*(1-m["agreement"])
        print(f"step={step+1:03d}/{GLOBAL_STEPS} loss={float(loss):.4f} agreement={m['agreement']:.4f} KL={m['mean_kl']:.5f}")
        if score<best_score:
            best_score=score; best_metrics=dict(m,step=step+1)
            best_state={n:p.detach().cpu().clone() for n,p in trainable}

if best_state:
    with torch.no_grad():
        for n,p in student.named_parameters():
            if n in best_state: p.copy_(best_state[n].to(device=p.device,dtype=p.dtype))
student.eval().requires_grad_(False)
final_metrics=validate(min(160,len(val_items)))
assert converted_full_attention_indices(student)==full_layers
print("best:",best_metrics,"| final:",final_metrics)


trainable params: 26,512,131
step=001/300 loss=0.4394 agreement=0.4896 KL=2.72862
step=050/300 loss=0.2470 agreement=0.3438 KL=0.21927
step=100/300 loss=0.7283 agreement=0.3021 KL=0.28300
step=150/300 loss=0.3000 agreement=0.4167 KL=0.18423
step=200/300 loss=0.7256 agreement=0.3333 KL=0.18025
step=250/300 loss=0.2133 agreement=0.5312 KL=0.15561
step=300/300 loss=0.1816 agreement=0.5000 KL=0.15278
best: {'agreement': 0.53125, 'mean_kl': 0.15561324279406108, 'step': 250} | final: {'agreement': 0.53125, 'mean_kl': 0.1377234716416785}


In [13]:
#@title 6. Fast Eval — held-out Student vs Teacher comparison
import time

FAST_EVAL_CASES = 96 #@param {type:"integer"}
FAST_EVAL_BATCH = 3 #@param {type:"integer"}
FAST_EVAL_WARMUP = 2 #@param {type:"integer"}
FAST_EVAL_REPEATS = 8 #@param {type:"integer"}
MIN_DECISION_AGREEMENT = 0.95 #@param {type:"number"}
MAX_MEAN_JS = 0.020 #@param {type:"number"}
MIN_ACTION_AGREEMENT = 0.95 #@param {type:"number"}
STRICT_FAST_EVAL = False #@param {type:"boolean"}

def _sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def _eval_forward(model, b):
    model.eval()
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        return model(**b)

def _js_divergence(p, q, eps=1e-8):
    p = p.float(); q = q.float()
    m = 0.5 * (p + q)
    lp = p.clamp_min(eps).log(); lq = q.clamp_min(eps).log(); lm = m.clamp_min(eps).log()
    return 0.5 * (p * (lp - lm)).sum(-1) + 0.5 * (q * (lq - lm)).sum(-1)

n_eval = min(max(1, int(FAST_EVAL_CASES)), len(val_items))
eval_batch = max(1, int(FAST_EVAL_BATCH))
decision_total = decision_agree = decision_top2 = 0
action_total = action_agree = 0
js_values, conf_gaps, teacher_conf, student_conf = [], [], [], []
disagreement_samples = []

teacher.eval(); student.eval()
for off in range(0, n_eval, eval_batch):
    n = min(eval_batch, n_eval - off)
    b = batch(val_items, off, n)
    tl, ta = _eval_forward(teacher, b)
    sl, sa = _eval_forward(student, b)

    marker_mask = b["marker_mask"].bool()
    tm = tl.float().masked_fill(~marker_mask, -1e9)
    sm = sl.float().masked_fill(~marker_mask, -1e9)
    tp = torch.softmax(tm, dim=-1)
    sp = torch.softmax(sm, dim=-1)

    t_choice = tm.argmax(-1)
    s_choice = sm.argmax(-1)
    same = (t_choice == s_choice)
    decision_agree += int(same.sum().item())
    decision_total += int(t_choice.numel())

    k2 = min(2, sm.shape[-1])
    s_top2 = torch.topk(sm, k=k2, dim=-1).indices
    decision_top2 += int((s_top2 == t_choice[:, None]).any(-1).sum().item())

    tc = tp.max(-1).values
    sc = sp.max(-1).values
    teacher_conf.extend(tc.detach().cpu().tolist())
    student_conf.extend(sc.detach().cpu().tolist())
    conf_gaps.extend((sc - tc).abs().detach().cpu().tolist())
    js_values.extend(_js_divergence(tp, sp).detach().cpu().tolist())

    if len(disagreement_samples) < 10:
        bad = (~same).nonzero(as_tuple=False).flatten().detach().cpu().tolist()
        for j in bad:
            item = val_items[off + j]
            disagreement_samples.append({
                "qid": str(item.get("qid", "")),
                "teacher_choice_index": int(t_choice[j].item()),
                "student_choice_index": int(s_choice[j].item()),
                "teacher_confidence": float(tc[j].item()),
                "student_confidence": float(sc[j].item()),
            })
            if len(disagreement_samples) >= 10:
                break

    if ta is not None and sa is not None:
        tact = ta.float().argmax(-1)
        sact = sa.float().argmax(-1)
        action_agree += int((tact == sact).sum().item())
        action_total += int(tact.numel())

def _latency_ms(model, probe):
    warmup = max(0, int(FAST_EVAL_WARMUP))
    repeats = max(1, int(FAST_EVAL_REPEATS))
    for _ in range(warmup):
        _eval_forward(model, probe)
    _sync_device()
    samples = []
    for _ in range(repeats):
        _sync_device()
        t0 = time.perf_counter()
        _eval_forward(model, probe)
        _sync_device()
        samples.append((time.perf_counter() - t0) * 1000.0)
    return samples

probe_n = min(eval_batch, n_eval)
probe = batch(val_items, 0, probe_n)
teacher_times = _latency_ms(teacher, probe)
student_times = _latency_ms(student, probe)
teacher_ms = float(np.median(teacher_times))
student_ms = float(np.median(student_times))
speedup = teacher_ms / max(student_ms, 1e-12)

decision_agreement = decision_agree / max(1, decision_total)
top2_recall = decision_top2 / max(1, decision_total)
action_agreement = (action_agree / action_total) if action_total else None
mean_js = float(np.mean(js_values)) if js_values else float("nan")
p95_js = float(np.percentile(js_values, 95)) if js_values else float("nan")
mean_conf_gap = float(np.mean(conf_gaps)) if conf_gaps else float("nan")

gates = {
    "decision_agreement": bool(decision_agreement >= MIN_DECISION_AGREEMENT),
    "mean_js": bool(mean_js <= MAX_MEAN_JS),
}
if action_agreement is not None:
    gates["action_agreement"] = bool(action_agreement >= MIN_ACTION_AGREEMENT)
quality_pass = bool(all(gates.values()))

fast_eval_report = {
    "cases": int(n_eval),
    "batch_size": int(probe_n),
    "sequence_length": int(probe["input_ids"].shape[1]),
    "decision_agreement": float(decision_agreement),
    "teacher_top1_in_student_top2": float(top2_recall),
    "action_head_agreement": None if action_agreement is None else float(action_agreement),
    "mean_js_divergence": float(mean_js),
    "p95_js_divergence": float(p95_js),
    "mean_abs_confidence_gap": float(mean_conf_gap),
    "teacher_mean_confidence": float(np.mean(teacher_conf)) if teacher_conf else None,
    "student_mean_confidence": float(np.mean(student_conf)) if student_conf else None,
    "teacher_latency_ms_median": float(teacher_ms),
    "student_latency_ms_median": float(student_ms),
    "teacher_items_per_second": float(1000.0 * probe_n / max(teacher_ms, 1e-12)),
    "student_items_per_second": float(1000.0 * probe_n / max(student_ms, 1e-12)),
    "student_speedup_vs_teacher": float(speedup),
    "teacher_parameters": int(sum(p.numel() for p in teacher.parameters())),
    "student_parameters": int(sum(p.numel() for p in student.parameters())),
    "quality_gates": gates,
    "quality_pass": quality_pass,
    "disagreement_samples": disagreement_samples,
}

print("\n=== FAST TEACHER ↔ STUDENT EVAL ===")
print(f"held-out cases          : {n_eval}")
print(f"decision agreement      : {decision_agreement:.2%}")
print(f"teacher top-1 in top-2  : {top2_recall:.2%}")
if action_agreement is not None:
    print(f"action-head agreement   : {action_agreement:.2%}")
print(f"mean / p95 JS divergence: {mean_js:.6f} / {p95_js:.6f}")
print(f"mean confidence |gap|   : {mean_conf_gap:.6f}")
print(f"teacher latency         : {teacher_ms:.2f} ms/batch | {fast_eval_report['teacher_items_per_second']:.1f} items/s")
print(f"student latency         : {student_ms:.2f} ms/batch | {fast_eval_report['student_items_per_second']:.1f} items/s")
print(f"student speedup         : {speedup:.2f}x")
print("quality gates           :", gates, "=>", "PASS" if quality_pass else "REVIEW")
if disagreement_samples:
    print("first disagreements     :", disagreement_samples[:3])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fast_eval_path = OUTPUT_DIR / "fast_eval_teacher_vs_student.json"
fast_eval_path.write_text(json.dumps(fast_eval_report, indent=2), encoding="utf-8")
print("saved report            :", fast_eval_path)

if STRICT_FAST_EVAL and not quality_pass:
    raise AssertionError(f"Fast evaluation failed quality gates: {gates}")



=== FAST TEACHER ↔ STUDENT EVAL ===
held-out cases          : 96
decision agreement      : 53.12%
teacher top-1 in top-2  : 93.75%
action-head agreement   : 100.00%
mean / p95 JS divergence: 0.064449 / 0.302550
mean confidence |gap|   : 0.184556
teacher latency         : 568.06 ms/batch | 5.3 items/s
student latency         : 592.14 ms/batch | 5.1 items/s
student speedup         : 0.96x
quality gates           : {'decision_agreement': False, 'mean_js': False, 'action_agreement': True} => REVIEW
first disagreements     : [{'qid': 'churn_risk', 'teacher_choice_index': 2, 'student_choice_index': 1, 'teacher_confidence': 0.5717459321022034, 'student_confidence': 0.3278917372226715}, {'qid': 'urgency', 'teacher_choice_index': 2, 'student_choice_index': 1, 'teacher_confidence': 0.4040926694869995, 'student_confidence': 0.3278917372226715}, {'qid': 'disposition', 'teacher_choice_index': 0, 'student_choice_index': 2, 'teacher_confidence': 0.9322540163993835, 'student_confidence': 0.4753668606

In [ ]:
#@title 7. Export COMPLETE standalone model + save/reload from Hugging Face
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_dir = OUTPUT_DIR / "hf_export"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

# Everything required for future inference is copied into one self-contained folder.
shutil.copytree(source_dir / "encoder", export_dir / "encoder")
shutil.copytree(source_dir / "tokenizer", export_dir / "tokenizer")

state = {k: v.detach().cpu().contiguous() for k, v in student.state_dict().items()}
save_file(state, str(export_dir / "model.safetensors"))

meta = {
    "format": "tinycenn-standalone-decision-v23",
    "architecture": "integrated_memory_v23",
    "source_model": SOURCE_MODEL,
    "head_layers": int(source_cfg.get("head_layers", 2)),
    "n_act": len(source_cfg.get("act_costs", {})) + 1,
    "max_len": int(source_cfg.get("max_len", MAX_LEN)),
    "head_max_len": int(source_cfg.get("head_max_len", 192)),
    "feature_dim": FEATURE_DIM,
    "local_kernel": LOCAL_KERNEL,
    "converted_full_attention_layers": full_layers,
    "all_full_attention_replaced": True,
    "laya_runtime_required": False,
    "local_steps_per_layer": LOCAL_STEPS,
    "global_finetune_steps": GLOBAL_STEPS,
    "validation": final_metrics,
    "fast_evaluation": globals().get("fast_eval_report"),
}
(export_dir / "standalone_config.json").write_text(
    json.dumps(meta, indent=2), encoding="utf-8"
)

# Ship the runtime itself so the HF repository remains usable without Laya.
shutil.copy2(Path(sd.__file__), export_dir / "standalone_decision.py")
(export_dir / "requirements.txt").write_text(
    "torch>=2.3\ntransformers>=4.45\nhuggingface_hub>=0.25\nsafetensors>=0.4\n",
    encoding="utf-8",
)

# A minimal standalone script is stored next to the weights on Hugging Face.
example_usage = r'''# Standalone Hugging Face usage
import sys
import torch
from huggingface_hub import snapshot_download

HF_REPO_ID = "__HF_REPO_ID__"
model_dir = snapshot_download(HF_REPO_ID)
sys.path.insert(0, model_dir)
from standalone_decision import load_standalone

device = "cuda" if torch.cuda.is_available() else "cpu"
runtime = load_standalone(model_dir, device=device)

result = runtime.decide(
    state={
        "speed_kmh": 61,
        "track_error": -0.16,
        "heading_error": -0.11,
        "curve": "left",
        "obstacle": False,
    },
    actions={
        "left": "steer left",
        "straight": "hold steering",
        "right": "steer right",
        "brake": "reduce speed",
    },
    instruction="Choose the safest next control action while keeping the vehicle near the center line.",
)
print(result)
'''.replace("__HF_REPO_ID__", HF_REPO_ID)
(export_dir / "example_usage.py").write_text(example_usage, encoding="utf-8")

fast_eval = globals().get("fast_eval_report") or {}
agreement_text = (
    f"{100.0 * fast_eval['decision_agreement']:.2f}%"
    if fast_eval.get("decision_agreement") is not None else "not run"
)
speedup_text = (
    f"{fast_eval['student_speedup_vs_teacher']:.2f}x"
    if fast_eval.get("student_speedup_vs_teacher") is not None else "not run"
)
model_card = f'''---
library_name: pytorch
tags:
- tinycenn
- decision-model
- game-control
- standalone
- attention-replacement
---

# Integrated Memory V2.3 — Standalone Decision Model

This repository contains the **complete exported model**, not an adapter. It does not require the `laya` Python package for inference.

## Architecture

- Source teacher: `{SOURCE_MODEL}`
- Replacement: TinyCeNN Integrated Memory V2.3
- All source `full_attention` layers replaced: `{full_layers}`
- Sliding-attention layers remain unchanged
- Export format: `tinycenn-standalone-decision-v23`

## Fast evaluation

- Teacher/student decision agreement: **{agreement_text}**
- Student speedup vs teacher: **{speedup_text}**

See `standalone_config.json` for the complete validation and fast-evaluation metadata.

## Install

```bash
pip install torch transformers huggingface_hub safetensors
```

## Load directly from Hugging Face

```python
import sys
import torch
from huggingface_hub import snapshot_download

model_dir = snapshot_download("{HF_REPO_ID}")
sys.path.insert(0, model_dir)
from standalone_decision import load_standalone

runtime = load_standalone(
    model_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

result = runtime.decide(
    state={{"speed_kmh": 61, "track_error": -0.16, "curve": "left"}},
    actions={{
        "left": "steer left",
        "straight": "hold steering",
        "right": "steer right",
        "brake": "reduce speed",
    }},
    instruction="Choose the safest next control action.",
)
print(result)
```

You can also run `example_usage.py` from this model repository.
'''
(export_dir / "README.md").write_text(model_card, encoding="utf-8")

# Validate the exact local export before uploading it.
runtime = load_standalone(str(export_dir), device=device)
smoke_state = {
    "speed_kmh": 61,
    "track_error": -0.16,
    "heading_error": -0.11,
    "curve": "left",
    "obstacle": False,
}
smoke_actions = {
    "left": "steer left",
    "straight": "hold steering",
    "right": "steer right",
    "brake": "reduce speed",
}
smoke_instruction = "Choose the safest next control action while keeping the vehicle near the center line."
local_result = runtime.decide(
    state=smoke_state,
    actions=smoke_actions,
    instruction=smoke_instruction,
)
print("local standalone smoke test:", local_result)
print("standalone export:", export_dir)

required_files = {
    "model.safetensors",
    "standalone_config.json",
    "standalone_decision.py",
    "requirements.txt",
    "README.md",
    "example_usage.py",
}
local_files = {p.relative_to(export_dir).as_posix() for p in export_dir.rglob("*") if p.is_file()}
missing_local = sorted(required_files - local_files)
assert not missing_local, f"Standalone export is incomplete: missing {missing_local}"
assert any(p.startswith("encoder/") for p in local_files), "encoder/ files missing"
assert any(p.startswith("tokenizer/") for p in local_files), "tokenizer/ files missing"

if UPLOAD_TO_HF:
    from huggingface_hub import HfApi, login

    token = os.environ.get("HF_TOKEN")
    if not token:
        login()

    api = HfApi(token=token)
    api.create_repo(
        HF_REPO_ID,
        repo_type="model",
        private=HF_PRIVATE,
        exist_ok=True,
    )
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(export_dir),
        commit_message="Upload complete standalone Integrated Memory V2.3 decision model",
    )

    # Verify persistence: required artifacts must really exist on the Hub.
    remote_files = set(api.list_repo_files(HF_REPO_ID, repo_type="model"))
    missing_remote = sorted(required_files - remote_files)
    assert not missing_remote, f"HF upload incomplete: missing {missing_remote}"
    assert any(p.startswith("encoder/") for p in remote_files), "HF encoder/ files missing"
    assert any(p.startswith("tokenizer/") for p in remote_files), "HF tokenizer/ files missing"

    # Critical final test: discard the local runtime and reload through the HF repo ID.
    hf_runtime = load_standalone(HF_REPO_ID, device=device, token=token)
    hf_result = hf_runtime.decide(
        state=smoke_state,
        actions=smoke_actions,
        instruction=smoke_instruction,
    )
    print("HF reload smoke test:", hf_result)
    print("HF model saved and verified:", f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("UPLOAD_TO_HF=False — model was exported locally only.")


## Later game-specific tuning

Fine-tune only the Integrated Memory cores + decision head on recorded `{state, candidate_actions, chosen_action}` tuples. Keep held-out tracks/seeds separate and gate deployment on action accuracy, collision rate, episode return, and inference latency.